## ⚔️ Benchmarking de Robustez Universal: Arena SOTA vs. Framework Híbrido

O pipeline de avaliação cross-dataset para comparação direta com o estado da arte (SOTA) foi consolidado de forma estritamente controlada, submetendo todas as arquiteturas aos mesmos conjuntos de teste fora de distribuição (*Out-of-Distribution* - OOD).

**Configuração do Ambiente e Protocolo de Execução:**
* **Infraestrutura Forense:** Kaggle Notebooks rodando sob ambiente isolado contendo instâncias aceleradas via hardware (NVIDIA Tesla T4 - 15GB VRAM efetivos).
* **Mecanismo de Persistência:** Sistema de checkpoints assíncronos via serialização binária (`.pkl`), garantindo o isolamento de memória e proteção contra fragmentação de alocação CUDA.
* **Métricas Computadas:** Acurácia Global (ACC), Área Sob a Curva ROC (AUC), escore F1-Score e detalhamento de desempenho por classe (*ACC_Real* vs. *ACC_Fake*) para detecção de vieses adaptativos.

**Baselines Avaliadas na Arena:**
1. **Cozzolino CLIP (latent10k_plus):** Extrator profundo operando estritamente sobre a representação do *Visual Encoder*.
2. **Cozzolino ResNet (Corvi2023):** Detector puramente convolucional especializado em assinaturas espaciais locais.
3. **Cozzolino Late Fusion:** Modelo combinatório tradicional baseado na média aritmética das probabilidades de saída dos dois encoders isolados.
4. **Nosso Modelo (Custom_CLIP):** Framework proposto baseado em fusão precoce (*Early Fusion*) de semântica latente bidirecional (via *Soft Prompt Tuning*) integrada a histogramas microtexturais espaciais (LBP).

### 📊 Tabela Consolidada de Resultados: SOTA vs. Framework Híbrido

Os resultados detalhados abaixo expõem as métricas brutas extraídas após a varredura completa das 54.000 imagens distribuídas nos quatro campos de batalha. O modelo híbrido proposto demonstra estabilidade superior e capacidade de generalização universal contra geradores não vistos.

| Dataset de Teste | Modelo Avaliado | ACC Global | AUC-ROC | F1-Score | ACC (Real) | ACC (Fake) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **AIGI-Holmes-Test** | Cozzolino CLIP | 75.53% | 0.9874 | 67.72% | **99.72%** | 51.34% |
| *(10.000 Amostras)* | Cozzolino ResNet | 87.00% | **0.9992** | 85.06% | 100.00% | 74.00% |
| | Cozzolino Late Fusion | 87.05% | 0.9984 | 85.12% | 100.00% | 74.10% |
| | **Nosso Modelo Híbrido** | **92.66%** | 0.9849 | **92.28%** | 97.62% | **87.70%** |
| **Artifact-Test** | Cozzolino CLIP | 65.79% | 0.7269 | 64.44% | 69.58% | 62.00% |
| *(10.000 Amostras)* | Cozzolino ResNet | 57.05% | 0.7544 | 26.59% | **98.54%** | 15.56% |
| | Cozzolino Late Fusion | 58.43% | 0.7729 | 30.77% | 98.38% | 18.48% |
| | **Nosso Modelo Híbrido** | **82.95%** | **0.9117** | **82.09%** | 87.74% | **78.16%** |
| **SID-Set-Test** | Cozzolino CLIP | 61.06% | 0.8619 | 39.29% | 96.92% | 25.20% |
| *(10.000 Amostras)* | Cozzolino ResNet | 64.39% | 0.9701 | 45.53% | 99.04% | 29.76% |
| | Cozzolino Late Fusion | 56.91% | 0.8635 | 24.87% | **99.60%** | 14.26% |
| | **Nosso Modelo Híbrido** | **98.35%** | **0.9991** | **98.37%** | 97.38% | **99.32%** |
| **Cozzolino-Test-Set**| Cozzolino CLIP | 70.04% | 0.8751 | 78.77% | 86.67% | 66.71% |
| *(24.000 Amostras)* | Cozzolino ResNet | 50.58% | 0.8767 | 57.89% | 99.58% | 40.77% |
| | Cozzolino Late Fusion | 53.07% | 0.9324 | 60.84% | **99.67%** | 43.75% |
| | **Nosso Modelo Híbrido** | **83.28%** | **0.9560** | **88.97%** | 95.00% | **80.94%** |
| 📊 **OVERALL** | Cozzolino CLIP | 68.61% | 0.8673 | 70.51% | 88.31% | 57.91% |
| *(Todas as Bases)* | Cozzolino ResNet | 61.08% | 0.8776 | 57.33% | 99.27% | 40.35% |
| | Cozzolino Late Fusion | 61.07% | 0.9175 | 57.28% | **99.40%** | 40.26% |
| | **Nosso Modelo (Híbrido)** | **87.75%** | **0.9615** | **89.90%** | 94.41% | **84.13%** |

**Principais Conclusões Estatísticas:**
* **Quebra de Viés Estático (*Asymmetry Bias*):** Os modelos SOTA baseados em ResNet e Fusão Tardia mostram uma falha crítica de generalização universal, apresentando taxas de acerto próximas a 100% em dados reais, mas despencando para patamares catastróficos (de 14% a 40%) na classe sintética (*ACC_Fake*), indicando uma tendência sistemática de classificar tudo como real. O framework proposto elimina esse comportamento, sustentando o equilíbrio em ambas as distribuições.
* **A Força do Descritor de Microtextura (LBP):** No cenário microscópico do `SID-Set-Test`, a incapacidade semântica dos modelos puros do CLIP e ResNet é escancarada (ficando próximos ao acaso com 56.91% de acurácia). A injeção precoce das características texturais do operador LBP permitiu ao nosso modelo resolver a base com **98.35% de ACC** e **0.9991 de AUC**.
* **Superioridade da Fusão Precoce:** Na consolidação macro (`OVERALL`), o nosso modelo superou a baseline de fusão de Cozzolino et al. por uma margem absoluta de **26.68% de acurácia**, validando empiricamente que unificar os tensores em baixo nível no espaço latente preserva propriedades forenses vitais que se perdem em fusões tardias baseadas em médias de probabilidade.

In [1]:
import os
import sys
import gc
import glob
import yaml
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from torchvision.transforms import CenterCrop, Resize, Compose, InterpolationMode

# Evita que o Kaggle encha o disco com cache desnecessário
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/datasets_cache"

# Clonar o seu repositório (que já contém as lógicas do Cozzolino integradas)
!git clone https://github.com/Jvfrostbr/ClipBased-SyntheticImageDetection.git
!pip install open_clip_torch scikit-learn albumentations

%cd /kaggle/working/ClipBased-SyntheticImageDetection
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Importações do seu modelo e das lógicas oficiais do Cozzolino
from networks.clip_custom_detector import CLIPImageClassifier
from utils.processing import make_normalize
from networks import create_architecture, load_weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Arena de Batalha Configurada! A utilizar: {DEVICE}")

class UniversalTestDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        amostra = self.samples[idx]
        try:
            image = Image.open(amostra['file_path']).convert('RGB')
            return image, amostra['label']
        except Exception:
            # Imagem de fallback vazia para não quebrar o loop em ficheiros corrompidos
            return Image.new('RGB', (224, 224)), amostra['label']

# Collate function personalizada para aplicar as transformações (que variam por modelo) "on the fly"
def collate_with_transform(batch, transform):
    images = []
    labels = []
    for img, label in batch:
        images.append(transform(img))
        labels.append(label)
    return torch.stack(images), torch.tensor(labels)

Cloning into 'ClipBased-SyntheticImageDetection'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 164 (delta 54), reused 73 (delta 36), pack-reused 58 (from 1)
Receiving objects: 100% (164/164), 627.00 KiB | 4.29 MiB/s, done.
Resolving deltas: 100% (65/65), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3

## **Mapeamento dos datasets**

In [2]:
PATHS_TEST = {
    "AIGI-Holmes-Test": "/kaggle/input/datasets/jvfrostbr/aigi-holmes-test",
    "Artifact-Test": "/kaggle/input/datasets/jvfrostbr/datasets-test/dataset_inferencia_estatico/artifact",
    "SID-Set-Test": "/kaggle/input/datasets/jvfrostbr/datasets-test/dataset_inferencia_estatico/sidset",
    "Cozzolino-Test-Set": "/kaggle/input/datasets/jvfrostbr/datasets-test/test_set_cozolino/test_set_cozolino/test_set"
}

datasets_mapeados = {}
print("📊 A mapear os campos de batalha com rastreio de origem...")

for nome_base, root_path in PATHS_TEST.items():
    amostras = []

    # Lógica de mapeamento exclusiva para a estrutura de pastas do Cozolino et al.
    if nome_base == "Cozzolino-Test-Set":
        pastas_modelos = glob.glob(os.path.join(root_path, "*"))
        for pasta in pastas_modelos:
            if os.path.isdir(pasta):
                nome_pasta = os.path.basename(pasta).lower()
                
                 # Regra: Pastas que começam com "real_" recebem label 0. O resto recebe 1
                label = 0 if nome_pasta.startswith("real_") else 1
                for img_path in glob.glob(os.path.join(pasta, "*.*")):
                    if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                        amostras.append({'file_path': img_path, 'label': label, 'subfolder': nome_pasta})
 
    # Lógica padrão para AIGI-Holmes, Artifact e SID-Set
    else:
        dir_real = os.path.join(root_path, "**", "0_real", "**", "*.*")
        dir_fake = os.path.join(root_path, "**", "1_fake", "**", "*.*")
        
        for path in glob.glob(dir_real, recursive=True):
            if path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                amostras.append({'file_path': path, 'label': 0, 'subfolder': '0_real'})
        for path in glob.glob(dir_fake, recursive=True):
            if path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                amostras.append({'file_path': path, 'label': 1, 'subfolder': '1_fake'})
            
    # O embaralhamento é mantido, mas as listas de predição respeitarão esta mesma ordem
    random.shuffle(amostras)
    datasets_mapeados[nome_base] = amostras
    print(f"✅ {nome_base}: {len(amostras)} imagens preparadas.")

📊 A mapear os campos de batalha com rastreio de origem...
✅ AIGI-Holmes-Test: 10000 imagens preparadas.
✅ Artifact-Test: 10000 imagens preparadas.
✅ SID-Set-Test: 10000 imagens preparadas.
✅ Cozzolino-Test-Set: 24000 imagens preparadas.


## **Definindo função para o carregamento do modelo cozolino**

In [3]:
# --- Lógica de Carregamento Oficial do Cozzolino (Kaggle Proof + PyTorch 2.6 Fix) ---
def load_cozzolino_model(model_name, weights_dir):
    import yaml
    
    padrao_busca = os.path.join(weights_dir, "**", model_name, "config.yaml")
    arquivos_config = glob.glob(padrao_busca, recursive=True)
    
    if not arquivos_config:
        raise FileNotFoundError(f"Erro: config.yaml do modelo '{model_name}' não encontrado em {weights_dir}")
        
    caminho_config_real = arquivos_config[0]
    diretorio_real_do_modelo = os.path.dirname(caminho_config_real)
    
    with open(caminho_config_real) as fid:
        data = yaml.load(fid, Loader=yaml.FullLoader)
        
    model_path = os.path.join(diretorio_real_do_modelo, data['weights_file'])
    
    # --- CORREÇÃO PYTORCH 2.6 ---
    # Criamos a arquitetura limpa primeiro
    model_limpo = create_architecture(data['arch'])
    
    # Carregamos os dados brutos ignorando a trava de segurança (weights_only=False)
    dados_carregados = torch.load(model_path, map_location='cpu', weights_only=False)
    
    # Injetamos os pesos na arquitetura da mesma forma que o __init__.py deles faria
    if 'model' in dados_carregados:
        estado_dict = dados_carregados['model']
        if ('module._conv_stem.weight' in estado_dict) or \
           ('module.conv1.weight' in estado_dict) or \
           ('module.backbone.conv1.weight' in estado_dict):
            estado_dict = {k.replace('module.', ''): v for k, v in estado_dict.items()}
    else:
        estado_dict = dados_carregados
        
    model_limpo.load_state_dict(estado_dict)
    model = model_limpo.to(DEVICE).eval()
    # -----------------------------
    
    transform_list = []
    patch_size = data['patch_size']
    norm_type = data['norm_type']
    
    if patch_size == 'Clip224':
        transform_list.extend([Resize(224, interpolation=InterpolationMode.BICUBIC), CenterCrop((224, 224))])
    elif isinstance(patch_size, (tuple, list)):
        transform_list.extend([Resize(*patch_size), CenterCrop(patch_size[0])])
    elif patch_size and patch_size > 0:
        transform_list.append(CenterCrop(patch_size))
        
    transform_list.append(make_normalize(norm_type))
    return model, Compose(transform_list)

## **Definindo funções auxiliares para extração de predições e cálculo de métricas**

In [4]:
import os
import pickle

# Criamos uma pasta segura para os checkpoints
DIR_CHECKPOINTS = "/kaggle/working/checkpoints"
os.makedirs(DIR_CHECKPOINTS, exist_ok=True)

def extrair_predicoes(modelo, transformacao, dataset, tipo_modelo="cozzolino", batch_size=64):
    from torchvision.transforms import Resize, CenterCrop
    import traceback
    
    tem_resize = False
    if hasattr(transformacao, 'transforms'):
        tem_resize = any(isinstance(t, (Resize, CenterCrop)) for t in transformacao.transforms)
        
    batch_seguro = batch_size if tem_resize else 1

    loader = DataLoader(
        dataset, batch_size=batch_seguro, shuffle=False, num_workers=4, pin_memory=True,
        collate_fn=lambda b: collate_with_transform(b, transformacao)
    )
    
    todas_probabilidades = []
    todos_alvos = []
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(tqdm(loader, desc=f"A extrair ({tipo_modelo})", leave=False)):
            try:
                images = images.to(DEVICE)
                outputs = modelo(images)
                
                if tipo_modelo == "cozzolino":
                    out_numpy = outputs.cpu().numpy()
                    if out_numpy.shape[1] == 1:
                        logits = out_numpy[:, 0]
                    elif out_numpy.shape[1] == 2:
                        logits = out_numpy[:, 1] - out_numpy[:, 0]
                    else:
                        logits = np.mean(out_numpy, (1, 2)) if len(out_numpy.shape) > 1 else out_numpy
                    probs = 1 / (1 + np.exp(-logits)) 
                elif tipo_modelo == "nosso":
                    probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                    
                if probs.ndim == 0:
                    probs = [probs.item()]
                    
                todas_probabilidades.extend(probs)
                todos_alvos.extend(labels.numpy())
                
            except Exception as e:
                print(f"\n⚠️ Imagem corrompida ou erro no lote {i}. Pulando... Erro: {e}")
                continue # Pula a imagem bixada e continua a vida!
            
            # --- PROTEÇÃO CONTRA O "PAU DOS 1000" ---
            if i > 0 and i % 500 == 0:
                torch.cuda.empty_cache() 
                
    return np.array(todas_probabilidades), np.array(todos_alvos)

def calcular_metricas_detalhadas(preds, targets):
    """ Calcula métricas Globais e por Classe (0=Real, 1=Fake) """
    preds_bin = (preds > 0.5).astype(int)
    targets = np.array(targets)
    
    # Proteções para divisão por zero caso o dataset só tenha uma classe
    mask_real = (targets == 0)
    mask_fake = (targets == 1)
    
    acc_global = accuracy_score(targets, preds_bin) if len(targets) > 0 else float('nan')
    f1_global = f1_score(targets, preds_bin) if len(targets) > 0 else float('nan')
    auc_global = roc_auc_score(targets, preds) if len(np.unique(targets)) > 1 else float('nan')
    
    acc_real = accuracy_score(targets[mask_real], preds_bin[mask_real]) if mask_real.sum() > 0 else float('nan')
    acc_fake = accuracy_score(targets[mask_fake], preds_bin[mask_fake]) if mask_fake.sum() > 0 else float('nan')
    
    f1_classes = f1_score(targets, preds_bin, average=None, labels=[0, 1])
    f1_real = f1_classes[0] if mask_real.sum() > 0 else float('nan')
    f1_fake = f1_classes[1] if mask_fake.sum() > 0 else float('nan')
    
    return {
        "ACC": acc_global, "AUC": auc_global, "F1": f1_global,
        "ACC_Real": acc_real, "ACC_Fake": acc_fake,
        "F1_Real": f1_real, "F1_Fake": f1_fake
    }

## **Baixando os pesos do modelo do Cozzolino via Git LFS**

In [5]:
!apt-get install git-lfs -y
!git lfs install

%cd /kaggle/working/ClipBased-SyntheticImageDetection
!git lfs pull




git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
Updated git hooks.
Git LFS initialized.
/kaggle/working/ClipBased-SyntheticImageDetection


## **Loop de Execução da comparação com os modelos SOTA:**

In [6]:
import os
import gc
import pickle
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# --- Configuração de Diretórios ---
DIR_WEIGHTS_COZZOLINO = "/kaggle/working/ClipBased-SyntheticImageDetection/weights"
DIR_WEIGHTS_CUSTOM_CLIP_FULL = "/kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v1.pth" 
DIR_CHECKPOINTS = "/kaggle/working/checkpoints"

# Garante que a pasta de checkpoints existe
os.makedirs(DIR_CHECKPOINTS, exist_ok=True)

# --- Estruturas de Armazenamento ---
resultados_globais = []
resultados_cozzolino_breakdown = []

# Dicionário para acumular predições e calcular o OVERALL no final
preds_overall = { 
    "Cozzolino_CLIP": {"p":[], "t":[]}, 
    "Cozzolino_ResNet": {"p":[], "t":[]}, 
    "Cozzolino_Fusao": {"p":[], "t":[]}, 
    "Nosso_Modelo": {"p":[], "t":[]} 
}

# --- Funções Auxiliares de Gerenciamento e Métricas ---
def carregar_ou_extrair(nome_base, nome_modelo, funcao_extrair, *args):
    """Gerencia os checkpoints no HD. Se o modelo já rodou nesta base, reaproveita."""
    nome_arquivo = f"{nome_base}_{nome_modelo.replace(' ', '_')}.pkl"
    caminho_ckpt = os.path.join(DIR_CHECKPOINTS, nome_arquivo)
    
    if os.path.exists(caminho_ckpt):
        print(f"♻️ Checkpoint encontrado! A carregar {nome_modelo} do HD...")
        with open(caminho_ckpt, 'rb') as f:
            dados = pickle.load(f)
        return dados['probs'], dados['targets']
    else:
        probs, targets = funcao_extrair(*args)
        with open(caminho_ckpt, 'wb') as f:
            pickle.dump({'probs': probs, 'targets': targets}, f)
        print(f"💾 Checkpoint de segurança guardado: {nome_arquivo}")
        return probs, targets

def processar_e_salvar_metricas(nome_modelo, probs, targets, amostras, dataset_nome):
    """Calcula as métricas detalhadas e popula os resultados da batalha."""
    mets = calcular_metricas_detalhadas(probs, targets)
    linha = {"Dataset": dataset_nome, "Modelo": nome_modelo}
    linha.update(mets)
    resultados_globais.append(linha)
    
    if dataset_nome == "Cozzolino-Test-Set":
        df_track = pd.DataFrame(amostras)
        df_track['probs'] = probs
        df_track['targets'] = targets
        
        for subfolder, group in df_track.groupby('subfolder'):
            mets_sub = calcular_metricas_detalhadas(group['probs'].values, group['targets'].values)
            resultados_cozzolino_breakdown.append({
                "Modelo": nome_modelo, 
                "Gerador/Origem": subfolder,
                "Tipo": "Real" if subfolder.startswith("real_") else "Fake",
                "ACC": mets_sub["ACC"], 
                "AUC": mets_sub["AUC"]
            })

# --- LOOP PRINCIPAL: A BATALHA DOS DATASETS ---
for nome_base, amostras in datasets_mapeados.items():
    if len(amostras) == 0: continue
    print(f"\n" + "="*60)
    print(f"⚔️ BATALHA NO DATASET: {nome_base.upper()}")
    print("="*60)
    
    dataset_obj = UniversalTestDataset(amostras)
    
    # --- 1. Cozzolino CLIP ---
    print("▶️ A executar: Cozzolino CLIP (latent10k_plus)")
    mod, trf = load_cozzolino_model("clipdet_latent10k_plus", DIR_WEIGHTS_COZZOLINO)
    probs_clip, targets = carregar_ou_extrair(nome_base, "Cozzolino CLIP", extrair_predicoes, mod, trf, dataset_obj, "cozzolino")
    processar_e_salvar_metricas("Cozzolino CLIP", probs_clip, targets, amostras, nome_base)
    preds_overall["Cozzolino_CLIP"]["p"].extend(probs_clip)
    preds_overall["Cozzolino_CLIP"]["t"].extend(targets)
    del mod; gc.collect(); torch.cuda.empty_cache()

    # --- 2. Corvi ResNet ---
    print("▶️ A executar: Cozzolino ResNet (Corvi2023)")
    mod, trf = load_cozzolino_model("Corvi2023", DIR_WEIGHTS_COZZOLINO)
    probs_corvi, targets = carregar_ou_extrair(nome_base, "Cozzolino ResNet", extrair_predicoes, mod, trf, dataset_obj, "cozzolino")
    processar_e_salvar_metricas("Cozzolino ResNet", probs_corvi, targets, amostras, nome_base)
    preds_overall["Cozzolino_ResNet"]["p"].extend(probs_corvi)
    preds_overall["Cozzolino_ResNet"]["t"].extend(targets)
    del mod; gc.collect(); torch.cuda.empty_cache()

    # --- 3. Fusão Tardia (CLIP + ResNet) ---
    print("▶️ A calcular: Fusão Tardia (CLIP + ResNet)")
    tamanho_minimo = min(len(probs_clip), len(probs_corvi))
    probs_fusao = (probs_clip[:tamanho_minimo] + probs_corvi[:tamanho_minimo]) / 2.0
    targets_fusao = targets[:tamanho_minimo]
    amostras_fusao = amostras[:tamanho_minimo]

    processar_e_salvar_metricas("Cozzolino Late Fusion", probs_fusao, targets_fusao, amostras_fusao, nome_base)
    preds_overall["Cozzolino_Fusao"]["p"].extend(probs_fusao)
    preds_overall["Cozzolino_Fusao"]["t"].extend(targets_fusao)

    # --- 4. Nosso Modelo Híbrido (Custom_CLIP) ---
    print("▶️ A executar: O Custom_CLIP (1_Completo)")
    nosso_modelo = CLIPImageClassifier(use_prompt=True, use_lbp=True, multiclass=False).to(DEVICE)
    pesos = torch.load(DIR_WEIGHTS_CUSTOM_CLIP_FULL, map_location=DEVICE, weights_only=False)
    nosso_modelo.load_state_dict(pesos['model_state_dict'] if 'model_state_dict' in pesos else pesos)
    nosso_modelo.eval()
    
    probs_nosso, targets_nosso = carregar_ou_extrair(nome_base, "Nosso_Modelo", extrair_predicoes, nosso_modelo, nosso_modelo.preprocess, dataset_obj, "nosso")
    processar_e_salvar_metricas("Nosso Modelo Híbrido", probs_nosso, targets_nosso, amostras, nome_base)
    preds_overall["Nosso_Modelo"]["p"].extend(probs_nosso)
    preds_overall["Nosso_Modelo"]["t"].extend(targets_nosso)
    del nosso_modelo; gc.collect(); torch.cuda.empty_cache()

# --- CÁLCULO DO OVERALL (MÉTRICAS ACUMULADAS) ---
for modelo_chave, listas in preds_overall.items():
    if len(listas["p"]) == 0: continue
    mets = calcular_metricas_detalhadas(np.array(listas["p"]), np.array(listas["t"]))
    linha = {"Dataset": "OVERALL (Todas as Bases)", "Modelo": modelo_chave.replace("_", " ")}
    linha.update(mets)
    resultados_globais.append(linha)

# --- CORREÇÃO E FORMATAÇÃO DAS TABELAS ---
def formatar_tabela(df):
    if df.empty: return df
    colunas_pct = [c for c in df.columns if "ACC" in c or "F1" in c]
    for col in colunas_pct:
        df[col] = df[col].apply(lambda x: f"{x*100:.2f}%" if pd.notnull(x) else "-")
    if "AUC" in df.columns:
        df["AUC"] = df["AUC"].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "-")
    return df

df_principal = formatar_tabela(pd.DataFrame(resultados_globais))
df_breakdown = formatar_tabela(pd.DataFrame(resultados_cozzolino_breakdown))

print("\n✅ Extração e cálculos finalizados com segurança! Execute a próxima célula para exibir as tabelas limpas.")


⚔️ BATALHA NO DATASET: AIGI-HOLMES-TEST
▶️ A executar: Cozzolino CLIP (latent10k_plus)


open_clip_pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

normalize CLIP


A extrair (cozzolino):   0%|          | 0/157 [00:00<?, ?it/s]

💾 Checkpoint de segurança guardado: AIGI-Holmes-Test_Cozzolino_CLIP.pkl
▶️ A executar: Cozzolino ResNet (Corvi2023)
normalize RESNET


...

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


A extrair (nosso):   0%|          | 0/375 [00:00<?, ?it/s]

💾 Checkpoint de segurança guardado: Cozzolino-Test-Set_Nosso_Modelo.pkl


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Extração e cálculos finalizados com segurança! Execute a próxima célula para exibir as tabelas limpas.


## **Análise de Desempenho e Comparação SOTA**

In [7]:
print("="*100)  
print("🏆 TABELA 1: RESULTADOS GLOBAIS E POR CLASSE (SOTA vs. HYBRID)")
print("="*100)
display(df_principal)
      
print("\n" + "="*100)
print("🔬 TABELA 2: ANÁLISE DETALHADA POR GERADOR (COZZOLINO TEST SET)")
print("="*100)
display(df_breakdown)

🏆 TABELA 1: RESULTADOS GLOBAIS E POR CLASSE (SOTA vs. HYBRID)


,Dataset,Modelo,ACC,AUC,F1,ACC_Real,ACC_Fake,F1_Real,F1_Fake
0,AIGI-Holmes-Test,Cozzolino CLIP,75.53%,0.9874,67.72%,99.72%,51.34%,80.30%,67.72%
1,AIGI-Holmes-Test,Cozzolino ResNet,87.00%,0.9992,85.06%,100.00%,74.00%,88.50%,85.06%
2,AIGI-Holmes-Test,Cozzolino Late Fusion,87.05%,0.9984,85.12%,100.00%,74.10%,88.53%,85.12%
3,AIGI-Holmes-Test,Nosso Modelo Híbrido,92.66%,0.9849,92.28%,97.62%,87.70%,93.01%,92.28%
4,Artifact-Test,Cozzolino CLIP,65.79%,0.7269,64.44%,69.58%,62.00%,67.04%,64.44%
5,Artifact-Test,Cozzolino ResNet,57.05%,0.7544,26.59%,98.54%,15.56%,69.64%,26.59%
6,Artifact-Test,Cozzolino Late Fusion,58.43%,0.7729,30.77%,98.38%,18.48%,70.30%,30.77%
7,Artifact-Test,Nosso Modelo Híbrido,82.95%,0.9117,82.09%,87.74%,78.16%,83.73%,82.09%
8,SID-Set-Test,Cozzolino CLIP,61.06%,0.8619,39.29%,96.92%,25.20%,71.34%,39.29%
9,SID-Set-Test,Cozzolino ResNet,64.39%,0.9701,45.53%,99.04%,29.76%,73.54%,45.53%



🔬 TABELA 2: ANÁLISE DETALHADA POR GERADOR (COZZOLINO TEST SET)


,Modelo,Gerador/Origem,Tipo,ACC,AUC
0,Cozzolino CLIP,biggan_256,Fake,71.80%,-
1,Cozzolino CLIP,biggan_512,Fake,42.20%,-
2,Cozzolino CLIP,dalle3_cocoval,Fake,46.90%,-
3,Cozzolino CLIP,dalle_2,Fake,19.40%,-
4,Cozzolino CLIP,ddpm,Fake,86.50%,-
...,...,...,...,...,...
163,Nosso Modelo Híbrido,stylegan2_ffhq_1024x1024,Fake,72.37%,-
164,Nosso Modelo Híbrido,stylegan2_ffhq_256x256,Fake,89.82%,-
165,Nosso Modelo Híbrido,stylegan2_lsundog_256x256,Fake,99.40%,-
166,Nosso Modelo Híbrido,stylegan3_r,Fake,86.60%,-
